In [6]:
# Install the core framework and the required chain/retrieval logic
# !pip install langchain==0.1.0 langchain-community==0.0.13 langchain-openai==0.0.2 faiss-cpu==1.7.4

In [7]:
# # Core framework and OpenAI integration
# !pip install -U langchain==1.2.9 langchain-openai==1.1.7


# # Community tools and Vector Store dependencies
# !pip install -U langchain-community faiss-cpu tiktoken

In [4]:
import os
import getpass
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [8]:
# --- 1. SETUP CREDENTIALS ---
# os.environ["OPENAI_API_KEY"] = "__"

# --- 2. GENERATE THE DATA FILE ---
content = """
LangChain Components:
1. Model I/O: Interface with language models including Prompts and Output Parsers.
2. Retrieval: Interface with application-specific data via loaders and vector stores.
3. Chains: Construct sequences of LLM calls.
4. Memory: Persist application state between runs.
"""
with open("langchain_info.txt", "w") as f:
    f.write(content)

# --- 3. LOAD & SPLIT TEXT DATA ---
loader = TextLoader("langchain_info.txt")
docs = loader.load()

In [9]:
# Recursive splitting keeps context meaningful
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
documents = text_splitter.split_documents(docs)

In [10]:
# --- 4. EMBEDDINGS & VECTOR STORE ---
embeddings = OpenAIEmbeddings()
# FAISS stores the vectors for similarity search
vectorstore = FAISS.from_documents(documents, embeddings)

In [11]:
# --- 5. SETUP RETRIEVER ---
retriever = vectorstore.as_retriever()

In [12]:
# --- 6. DEFINE LLM & PROMPT ---
llm = ChatOpenAI(model="gpt-4o", temperature=0)

template = """Use the following pieces of retrieved context to answer the question.

Context: {context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)


In [13]:
# --- 7. CREATE RAG CHAIN (Alternative approach using LCEL) ---
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [17]:
rag_chain = (
   {"context": retriever | format_docs, "question": RunnablePassthrough()}
   | prompt
   | llm
   | StrOutputParser()
)

In [19]:
response = rag_chain.invoke("What are the 4 components of langchain concisely??")

In [20]:
print(response)

The four components of LangChain are Model I/O, Retrieval, Chains, and Memory.
